In [1]:
from pathlib import Path
import sys
import json
import pickle

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

CLASSES = ["akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"]
LABEL2IDX = {c: i for i, c in enumerate(CLASSES)}
IDX2LABEL = {i: c for c, i in LABEL2IDX.items()}
PROB_COLS = [f"prob_{c}" for c in CLASSES]

def find_project_root(start=Path.cwd()):
    p = Path(start).resolve()
    while p != p.parent:
        if (p / "EFFNet").exists() and (p / "preprocess").exists():
            return p
        p = p.parent
    raise RuntimeError("Cannot find project root containing EFFNet/ and preprocess/")

PROJECT_ROOT = find_project_root()
EFFNET_ROOT = PROJECT_ROOT / "EFFNet"
SRC_DIR = EFFNET_ROOT / "src"

sys.path.insert(0, str(SRC_DIR))

LINE1_DIR = EFFNET_ROOT / "experiments/line1_nguyen2022_soft_attention/outputs_effnetv2_m_softattention_metadata_finetune_classifier"
LINE2_STEPA_DIR = EFFNET_ROOT / "experiments/line2_effnet_feature_fusion_rf/stepA_effnetv2_m_metadata_backbone/outputs_effnetv2_m_metadata_finetune_classifier"

ENSEMBLE_DIR = EFFNET_ROOT / "experiments/ensemble_line1_line2_stepA"
ENSEMBLE_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("LINE1_DIR:", LINE1_DIR)
print("LINE2_STEPA_DIR:", LINE2_STEPA_DIR)

PROJECT_ROOT: /mnt/e/BIP/Proj_x
LINE1_DIR: /mnt/e/BIP/Proj_x/EFFNet/experiments/line1_nguyen2022_soft_attention/outputs_effnetv2_m_softattention_metadata_finetune_classifier
LINE2_STEPA_DIR: /mnt/e/BIP/Proj_x/EFFNet/experiments/line2_effnet_feature_fusion_rf/stepA_effnetv2_m_metadata_backbone/outputs_effnetv2_m_metadata_finetune_classifier


In [2]:
def compute_metrics_from_idx(y_true_idx, y_pred_idx):
    return {
        "accuracy": accuracy_score(y_true_idx, y_pred_idx),
        "balanced_accuracy": balanced_accuracy_score(y_true_idx, y_pred_idx),
        "precision_macro": precision_score(y_true_idx, y_pred_idx, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true_idx, y_pred_idx, average="macro", zero_division=0),
        "f1_macro": f1_score(y_true_idx, y_pred_idx, average="macro", zero_division=0),
    }

def evaluate_prob_df(df, prob_cols=PROB_COLS):
    y_true = df["true_label"].map(LABEL2IDX).to_numpy()
    probs = df[prob_cols].to_numpy()
    y_pred = probs.argmax(axis=1)
    metrics = compute_metrics_from_idx(y_true, y_pred)
    pred_labels = [IDX2LABEL[i] for i in y_pred]
    return metrics, y_true, y_pred, pred_labels

def save_eval_outputs(df, output_dir, prefix):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    metrics, y_true, y_pred, pred_labels = evaluate_prob_df(df)

    out = df.copy()
    out["pred_label"] = pred_labels
    out["correct"] = out["true_label"] == out["pred_label"]
    out.to_csv(output_dir / f"{prefix}_predictions.csv", index=False)

    pd.DataFrame([metrics]).to_csv(output_dir / f"{prefix}_metrics.csv", index=False)

    report = classification_report(
        y_true,
        y_pred,
        target_names=CLASSES,
        output_dict=True,
        zero_division=0,
    )
    pd.DataFrame(report).transpose().to_csv(output_dir / f"{prefix}_per_class_metrics.csv")

    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(CLASSES))))
    pd.DataFrame(cm, index=CLASSES, columns=CLASSES).to_csv(output_dir / f"{prefix}_confusion_matrix.csv")

    return metrics

In [3]:
line1_pred = pd.read_csv(LINE1_DIR / "predictions.csv")
line2_pred = pd.read_csv(LINE2_STEPA_DIR / "predictions.csv")

keep_cols = ["image_id", "true_label"] + PROB_COLS

line1 = line1_pred[keep_cols].copy()
line2 = line2_pred[keep_cols].copy()

merged = line1.merge(
    line2,
    on=["image_id", "true_label"],
    suffixes=("_line1", "_line2"),
    validate="one_to_one",
)

print("merged:", merged.shape)
print(merged.head())

merged: (1002, 16)
       image_id true_label  prob_akiec_line1  prob_bcc_line1  prob_bkl_line1  \
0  ISIC_0032195         nv      4.286195e-08    7.479645e-07    6.504592e-07   
1  ISIC_0033670        mel      4.993071e-07    7.972827e-07    9.315048e-05   
2  ISIC_0034181         nv      2.070600e-08    3.115099e-08    2.873007e-07   
3  ISIC_0032506         nv      2.324043e-21    5.079735e-23    3.100549e-17   
4  ISIC_0026216         nv      3.341725e-05    8.936816e-05    3.383723e-03   

   prob_df_line1  prob_mel_line1  prob_nv_line1  prob_vasc_line1  \
0   4.903299e-10    2.227070e-04       0.999776     6.432309e-10   
1   2.290969e-08    9.885113e-01       0.011394     1.775823e-08   
2   1.833602e-09    7.209870e-03       0.992790     4.341430e-09   
3   2.897068e-20    2.441848e-19       1.000000     2.586720e-21   
4   1.420401e-04    4.268644e-01       0.569476     1.110101e-05   

   prob_akiec_line2  prob_bcc_line2  prob_bkl_line2  prob_df_line2  \
0      5.249064e-06  

- 搜索 ensemble 权重 alpha

In [5]:
def build_ensemble_df(merged, alpha):
    out = merged[["image_id", "true_label"]].copy()

    p1 = merged[[f"{c}_line1" for c in PROB_COLS]].to_numpy()
    p2 = merged[[f"{c}_line2" for c in PROB_COLS]].to_numpy()

    fused = alpha * p1 + (1 - alpha) * p2
    fused = fused / fused.sum(axis=1, keepdims=True)

    for i, col in enumerate(PROB_COLS):
        out[col] = fused[:, i]

    return out

rows = []

for alpha in np.linspace(0.0, 1.0, 21):
    ens_df = build_ensemble_df(merged, alpha)
    metrics, _, _, _ = evaluate_prob_df(ens_df)

    rows.append({
        "alpha_line1": alpha,
        "alpha_line2_stepA": 1 - alpha,
        **metrics,
    })

search_df = pd.DataFrame(rows).sort_values("f1_macro", ascending=False)
search_df

,alpha_line1,alpha_line2_stepA,accuracy,balanced_accuracy,precision_macro,recall_macro,f1_macro
14,0.70,0.30,0.918164,0.904541,0.877974,0.904541,0.888871
12,0.60,0.40,0.917166,0.903242,0.877277,0.903242,0.887883
13,0.65,0.35,0.917166,0.903242,0.877277,0.903242,0.887883
9,0.45,0.55,0.920160,0.903476,0.875009,0.903476,0.886898
8,0.40,0.60,0.918164,0.903050,0.874697,0.903050,0.886708
7,0.35,0.65,0.917166,0.903911,0.872219,0.903911,0.885944
6,0.30,0.70,0.915170,0.903497,0.872389,0.903497,0.885895
1,0.05,0.95,0.909182,0.897471,0.874976,0.897471,0.885547
10,0.50,0.50,0.917166,0.904351,0.872118,0.904351,0.885150
5,0.25,0.75,0.914172,0.902210,0.871847,0.902210,0.884987


In [6]:
best = search_df.iloc[0]
best_alpha = float(best["alpha_line1"])

print("best alpha:", best_alpha)
print(best)

best_ens_df = build_ensemble_df(merged, best_alpha)
metrics = save_eval_outputs(best_ens_df, ENSEMBLE_DIR, prefix=f"ensemble_alpha_{best_alpha:.2f}")

with open(ENSEMBLE_DIR / "ensemble_search_results.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "best_alpha_line1": best_alpha,
            "best_alpha_line2_stepA": 1 - best_alpha,
            "best_metrics": metrics,
            "all_results": search_df.to_dict(orient="records"),
        },
        f,
        indent=2,
    )

metrics

best alpha: 0.7000000000000001
alpha_line1          0.700000
alpha_line2_stepA    0.300000
accuracy             0.918164
balanced_accuracy    0.904541
precision_macro      0.877974
recall_macro         0.904541
f1_macro             0.888871
Name: 14, dtype: float64


{'accuracy': 0.9181636726546906,
 'balanced_accuracy': np.float64(0.9045405409918429),
 'precision_macro': 0.8779736425774438,
 'recall_macro': 0.9045405409918429,
 'f1_macro': 0.8888711611221883}

In [7]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image, ImageOps
from tqdm import tqdm

from train_SA import EfficientNetV2SoftAttentionMetadataClassifier
from train_effnet_metadat import EfficientNetV2MetadataClassifier

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

/home/jamyoung/miniconda3/envs/yolov3/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device(type='cuda')

In [8]:
def load_metadata_features_for_split(split_csv, preprocessor_pkl):
    df = pd.read_csv(split_csv).copy()

    with open(preprocessor_pkl, "rb") as f:
        prep = pickle.load(f)

    age_median = prep["age_median"]
    scaler = prep["scaler"]
    encoder = prep["encoder"]

    df["age"] = df["age"].fillna(age_median)

    for col in ["sex", "localization"]:
        df[col] = df[col].fillna("unknown").astype(str)

    age = scaler.transform(df[["age"]])
    cat = encoder.transform(df[["sex", "localization"]])

    meta = np.concatenate([age, cat], axis=1).astype("float32")

    return df, meta

def apply_tta_pil(image, mode):
    if mode == "orig":
        return image
    if mode == "hflip":
        return ImageOps.mirror(image)
    if mode == "vflip":
        return ImageOps.flip(image)
    if mode == "hvflip":
        return ImageOps.flip(ImageOps.mirror(image))
    raise ValueError(f"Unknown TTA mode: {mode}")

class HAMTTADataset(Dataset):
    def __init__(self, dataframe, metadata_array, image_size=300, tta_modes=None):
        self.df = dataframe.reset_index(drop=True)
        self.metadata_array = metadata_array
        self.tta_modes = tta_modes or ["orig", "hflip", "vflip", "hvflip"]

        self.post_tfm = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225],
            ),
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["image_path"]).convert("RGB")

        images = []
        for mode in self.tta_modes:
            aug = apply_tta_pil(image, mode)
            images.append(self.post_tfm(aug))

        images = torch.stack(images, dim=0)
        meta = torch.tensor(self.metadata_array[idx], dtype=torch.float32)
        label = torch.tensor(int(row["label"]), dtype=torch.long)
        image_id = row["image_id"]

        return images, meta, label, image_id

In [9]:
def load_line1_model():
    ckpt_path = LINE1_DIR / "best_effnetv2_softattention_metadata_classifier.pth"
    ckpt = torch.load(ckpt_path, map_location=device)

    model = EfficientNetV2SoftAttentionMetadataClassifier(
        metadata_dim=int(ckpt["metadata_dim"]),
        model_name=ckpt.get("model_name", "tf_efficientnetv2_m"),
        num_classes=len(CLASSES),
        pretrained=False,
        num_heads=int(ckpt.get("num_heads", 16)),
    )

    model.load_state_dict(ckpt["model_state_dict"])
    model.to(device)
    model.eval()

    return model, ckpt

def load_line2_stepA_model():
    ckpt_path = LINE2_STEPA_DIR / "best_effnetv2_metadata_classifier.pth"
    ckpt = torch.load(ckpt_path, map_location=device)

    model = EfficientNetV2MetadataClassifier(
        metadata_dim=int(ckpt["metadata_dim"]),
        model_name=ckpt.get("model_name", "tf_efficientnetv2_m"),
        num_classes=len(CLASSES),
        pretrained=False,
    )

    model.load_state_dict(ckpt["model_state_dict"])
    model.to(device)
    model.eval()

    return model, ckpt

line1_model, line1_ckpt = load_line1_model()
line2_model, line2_ckpt = load_line2_stepA_model()

print("line1 best:", line1_ckpt.get("best_epoch"), line1_ckpt.get("best_val_f1_macro"))
print("line2 image_size:", line2_ckpt.get("image_size"))

/tmp/ipykernel_219490/1826502506.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(ckpt_path, map_location=device)
/tmp/ipykernel_219490/1826502506.py:21

line1 best: 57 0.8784032988204317
line2 image_size: 300


In [10]:
@torch.no_grad()
def predict_tta(model, loader, desc="TTA"):
    model.eval()

    all_probs = []
    all_labels = []
    all_image_ids = []

    for images_tta, metas, labels, image_ids in tqdm(loader, desc=desc):
        # images_tta: [B, T, C, H, W]
        bsz, num_tta, c, h, w = images_tta.shape

        images = images_tta.view(bsz * num_tta, c, h, w).to(device, non_blocking=True)
        metas = metas.to(device, non_blocking=True)
        metas = metas.repeat_interleave(num_tta, dim=0)

        logits = model(images, metas)
        probs = torch.softmax(logits, dim=1)
        probs = probs.view(bsz, num_tta, len(CLASSES)).mean(dim=1)

        all_probs.append(probs.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_image_ids.extend(list(image_ids))

    all_probs = np.concatenate(all_probs, axis=0)

    out = pd.DataFrame({
        "image_id": all_image_ids,
        "true_label": [IDX2LABEL[int(i)] for i in all_labels],
    })

    for i, cls in enumerate(CLASSES):
        out[f"prob_{cls}"] = all_probs[:, i]

    metrics, _, _, pred_labels = evaluate_prob_df(out)
    out["pred_label"] = pred_labels
    out["correct"] = out["true_label"] == out["pred_label"]

    return out, metrics

In [11]:
TTA_MODES = ["orig", "hflip", "vflip", "hvflip"]
BATCH_SIZE = 8
NUM_WORKERS = 0

line1_val_df, line1_val_meta = load_metadata_features_for_split(
    LINE1_DIR / "val_split.csv",
    LINE1_DIR / "metadata_preprocessor.pkl",
)

line2_val_df, line2_val_meta = load_metadata_features_for_split(
    LINE2_STEPA_DIR / "val_split.csv",
    LINE2_STEPA_DIR / "metadata_preprocessor.pkl",
)

line1_ds = HAMTTADataset(
    line1_val_df,
    line1_val_meta,
    image_size=int(line1_ckpt.get("image_size", 300)),
    tta_modes=TTA_MODES,
)

line2_ds = HAMTTADataset(
    line2_val_df,
    line2_val_meta,
    image_size=int(line2_ckpt.get("image_size", 300)),
    tta_modes=TTA_MODES,
)

line1_loader = DataLoader(
    line1_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

line2_loader = DataLoader(
    line2_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

line1_tta_df, line1_tta_metrics = predict_tta(line1_model, line1_loader, desc="line1 TTA")
line2_tta_df, line2_tta_metrics = predict_tta(line2_model, line2_loader, desc="line2 stepA TTA")

print("line1 TTA:", line1_tta_metrics)
print("line2 stepA TTA:", line2_tta_metrics)

line1_tta_df.to_csv(ENSEMBLE_DIR / "line1_tta_predictions.csv", index=False)
line2_tta_df.to_csv(ENSEMBLE_DIR / "line2_stepA_tta_predictions.csv", index=False)

line2 stepA TTA: 100%|██████████| 126/126 [00:47<00:00,  2.64it/s]

line1 TTA: {'accuracy': 0.9121756487025948, 'balanced_accuracy': np.float64(0.8959013290739424), 'precision_macro': 0.8750736274467307, 'recall_macro': 0.8959013290739424, 'f1_macro': 0.8825424672358897}
line2 stepA TTA: {'accuracy': 0.9131736526946108, 'balanced_accuracy': np.float64(0.8919824328007274), 'precision_macro': 0.8716935334523723, 'recall_macro': 0.8919824328007274, 'f1_macro': 0.8809672429126196}


In [12]:
line1_tta = line1_tta_df[["image_id", "true_label"] + PROB_COLS].copy()
line2_tta = line2_tta_df[["image_id", "true_label"] + PROB_COLS].copy()

tta_merged = line1_tta.merge(
    line2_tta,
    on=["image_id", "true_label"],
    suffixes=("_line1", "_line2"),
    validate="one_to_one",
)

rows = []

for alpha in np.linspace(0.0, 1.0, 21):
    ens_df = build_ensemble_df(tta_merged, alpha)
    metrics, _, _, _ = evaluate_prob_df(ens_df)

    rows.append({
        "alpha_line1_tta": alpha,
        "alpha_line2_stepA_tta": 1 - alpha,
        **metrics,
    })

tta_search_df = pd.DataFrame(rows).sort_values("f1_macro", ascending=False)
tta_search_df

,alpha_line1_tta,alpha_line2_stepA_tta,accuracy,balanced_accuracy,precision_macro,recall_macro,f1_macro
10,0.50,0.50,0.926148,0.914284,0.893840,0.914284,0.901591
12,0.60,0.40,0.925150,0.914048,0.890496,0.914048,0.899870
11,0.55,0.45,0.925150,0.914060,0.889000,0.914060,0.899253
8,0.40,0.60,0.922156,0.910833,0.886632,0.910833,0.897334
13,0.65,0.35,0.922156,0.911249,0.886882,0.911249,0.896666
9,0.45,0.55,0.923154,0.911046,0.882315,0.911046,0.894488
14,0.70,0.30,0.921158,0.908877,0.883752,0.908877,0.893824
16,0.80,0.20,0.921158,0.906288,0.884536,0.906288,0.892987
15,0.75,0.25,0.921158,0.907363,0.883115,0.907363,0.892842
7,0.35,0.65,0.918164,0.905661,0.881442,0.905661,0.892114


In [13]:
best_tta = tta_search_df.iloc[0]
best_tta_alpha = float(best_tta["alpha_line1_tta"])

print("best TTA alpha:", best_tta_alpha)
print(best_tta)

best_tta_ens_df = build_ensemble_df(tta_merged, best_tta_alpha)
tta_metrics = save_eval_outputs(
    best_tta_ens_df,
    ENSEMBLE_DIR,
    prefix=f"tta_ensemble_alpha_{best_tta_alpha:.2f}",
)

with open(ENSEMBLE_DIR / "tta_ensemble_search_results.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "tta_modes": TTA_MODES,
            "best_alpha_line1_tta": best_tta_alpha,
            "best_alpha_line2_stepA_tta": 1 - best_tta_alpha,
            "best_metrics": tta_metrics,
            "line1_tta_metrics": line1_tta_metrics,
            "line2_stepA_tta_metrics": line2_tta_metrics,
            "all_results": tta_search_df.to_dict(orient="records"),
        },
        f,
        indent=2,
    )

tta_metrics

best TTA alpha: 0.5
alpha_line1_tta          0.500000
alpha_line2_stepA_tta    0.500000
accuracy                 0.926148
balanced_accuracy        0.914284
precision_macro          0.893840
recall_macro             0.914284
f1_macro                 0.901591
Name: 10, dtype: float64


{'accuracy': 0.9261477045908184,
 'balanced_accuracy': np.float64(0.9142843716614208),
 'precision_macro': 0.8938397581254724,
 'recall_macro': 0.9142843716614208,
 'f1_macro': 0.9015905985546141}